# Demo de solicitudes de vacaciones y permisos

Conecta a PostgreSQL en Supabase para consultar y registrar datos ficticios. No introduzcas datos reales de trabajadores ni compartas credenciales. Prepara la base ejecutando `sql/schema.sql` en Supabase SQL Editor.

In [ ]:
%pip -q install 'psycopg[binary]' pandas ipywidgets
import getpass
import pandas as pd
import psycopg

# Pega la cadena PostgreSQL de Supabase cuando aparezca el aviso.
# Usa Session pooler si la conexión directa no funciona desde Colab.
conninfo = getpass.getpass('Cadena PostgreSQL de Supabase (entrada oculta): ')
conn = psycopg.connect(conninfo, connect_timeout=20)
print('Conexión establecida.')

In [ ]:
consulta = '''
select area, estado, tipo, cantidad
from rrhh_demo.resumen_por_estado
order by area, estado, tipo;
'''
resumen = pd.read_sql_query(consulta, conn)
display(resumen)

if not resumen.empty:
    grafico = resumen.groupby('estado')['cantidad'].sum().sort_values(ascending=False)
    ax = grafico.plot(kind='bar', title='Solicitudes ficticias por estado',
                      xlabel='Estado', ylabel='Cantidad', color='#2563eb', rot=0)
    ax.figure.tight_layout()

In [ ]:
# Listado de solicitudes de ejemplo con códigos ficticios.
solicitudes = pd.read_sql_query('''
select s.codigo_trabajador, a.nombre as area, s.tipo,
       s.fecha_inicio, s.fecha_fin, s.estado
from rrhh_demo.solicitudes s
join rrhh_demo.areas a on a.id = s.area_id
order by s.fecha_inicio;
''', conn)
display(solicitudes)

## Registrar una solicitud ficticia

Usa un código como `DEMO-004`; no escribas nombres ni documentos de identidad. El formulario guarda la solicitud en Supabase.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

areas = pd.read_sql_query('select id, nombre from rrhh_demo.areas order by nombre', conn)
area_widget = widgets.Dropdown(options=[(r['nombre'], int(r['id'])) for _, r in areas.iterrows()], description='Área:')
codigo_widget = widgets.Text(value='DEMO-004', description='Código:', placeholder='DEMO-004')
tipo_widget = widgets.Dropdown(options=['vacaciones', 'permiso'], description='Tipo:')
inicio_widget = widgets.DatePicker(description='Desde:')
fin_widget = widgets.DatePicker(description='Hasta:')
guardar_widget = widgets.Button(description='Registrar solicitud', button_style='primary')
salida_widget = widgets.Output()

def registrar_solicitud(_):
    with salida_widget:
        clear_output()
        codigo = codigo_widget.value.strip().upper()
        if not codigo.startswith('DEMO-'):
            print('Usa un código ficticio que empiece con DEMO-.')
            return
        if inicio_widget.value is None or fin_widget.value is None:
            print('Selecciona ambas fechas.')
            return
        if fin_widget.value < inicio_widget.value:
            print('La fecha final no puede ser anterior a la inicial.')
            return
        try:
            with conn.cursor() as cur:
                cur.execute('''insert into rrhh_demo.solicitudes
                    (codigo_trabajador, area_id, tipo, fecha_inicio, fecha_fin, estado, observacion_demo)
                    values (%s, %s, %s, %s, %s, 'pendiente', 'Registro ficticio desde Colab') returning id''',
                    (codigo, area_widget.value, tipo_widget.value, inicio_widget.value, fin_widget.value))
                nuevo_id = cur.fetchone()[0]
            conn.commit()
            print(f'Solicitud ficticia registrada con código {nuevo_id}.')
        except Exception as error:
            conn.rollback()
            print(f'No se pudo guardar la solicitud: {error}')

guardar_widget.on_click(registrar_solicitud)
display(widgets.VBox([codigo_widget, area_widget, tipo_widget, inicio_widget, fin_widget, guardar_widget, salida_widget]))

## Cerrar la conexión

Ejecuta esta celda al terminar. Para seguir usando el notebook, vuelve a ejecutar la celda de conexión.

In [ ]:
conn.close()
print('Conexión cerrada.')